# Pressure versus capillary temperature
Start with the capillary at ambient temperature. Do not leave the experiment unattended. During the run the notebook asks Windows to keep the PC and display awake; it cannot block a manual sleep, a closed lid or the power button, so also disable sleep/hibernation in the power plan. If the PC still sleeps, or no sample starts for more than **60 s** (`OBSERVATION_STALL_S`), heating stops with an explicit message. Close Explorer, CGC utilities and all older instrument-owning kernels/processes. Use a fresh Windows kernel and the `esi/` and `tpg366/` plugins beside `notebooks/` (or set `PLUGINS_DIR`); the runtime files' SHA-256 are recorded in the run metadata. Verify COM ports and TPG USB baud before running the single cell. **`ARM_HEATING=True` by default: running the cell starts the real 175 °C protocol.** Set it to `False` for a no-hardware check. This is an **active** experiment; ESI uses its standard connection at 230400 baud. The capillary, seals and fluid must be suitable for the proposed temperatures; software tests do not qualify the hardware at 175 °C.

- **Baseline:** 60 s, heater and HV verified OFF.
- **Heating:** all stages **30, 40, …, 170, then 175 °C**. An initial temperature above 30 °C refuses heating: let the instrument cool before starting this full ascending protocol; no stage is skipped. Fixed ceilings **22 V / 10 A / 50 W**, bounded to fresh hardware maxima. Native applied echoes and independent readback must stay within both bounds; an excessive value is refused, not tolerated or silently clamped. No automatic increase or profile load. The reported maximum temperature must support 175 °C. Each integer stage target is exactly representable by the matched DLL; its applied echo must equal the requested target, independently of the measured-temperature stability band.
- **Each stage:** a full **60 s** within **±0.2 °C**, absolute OLS slope **<0.1 °C/min**, then **300 s continuously qualified** observation. First qualification must finish within **600 s**; the absolute stage deadline is **900 s** from the command. A temperature-observation gap over **10 s** (`MAX_SAMPLE_GAP_S`) or loss of qualification resets the observation, never the deadline. This does not establish thermal equilibrium or pressure equilibrium.
- **175 °C hold is attempted:** accepted band **174.8–175.0 °C**. A measured temperature **>175 °C** stops heating and leaves the stage incomplete. Sampled software checks cannot prevent overshoot between reads or after OFF; they are not an independent hardware limiter.
- **Cooling observation:** after verified heater OFF, record fresh temperature and pressures for **900 s**, then perform verified ESI shutdown/discharge. Stop and faults take priority. This is **not proof of being cold or safe to touch**. Electrical limits remain configured; no NVM save is issued.

Temperature and pressure timestamps identify sequential software observations, not synchronized or certified-fresh ADC conversions. At least one of six pressures must remain valid; other gauges retain their exact status and NaN. No pressure-equilibrium threshold or vacuum interlock is inferred. CSV retains nonpositive pressures; logarithmic plots omit them. Partial CSV, metadata and two PNGs are saved under `pressure_temperature_runs/`. Pressure may continue during final discharge; unavailable temperature is never copied forward.

**Stop** requests cooperative OFF after the current call; it never starts concurrent cleanup. A blocked or unknown transport stays unusable and its shutdown stays unconfirmed. Both ESI notebooks share a per-user COM ownership guard and consult historical heater/PT markers. Recovery requires a run-specific operator declaration of physical safety and termination of all former owners; original reports and uncertainty are archived, not rewritten as confirmed shutdown. Do not delete markers or restart a kernel to bypass them. Other accounts, unknown port aliases and arbitrary legacy output locations are not automatically covered. `ARM_HEATING=False` accesses no hardware and does not turn an existing heater OFF.


In [ ]:
# Close Explorer first. Approved experiment ceilings; these remain configured after OFF.
ARM_HEATING = True  # Running this cell starts the approved real 175 °C protocol.
PLUGINS_DIR = None  # Optional: folder containing esi/ and tpg366/; otherwise found nearby.
ESI_COM = 16
ESI_BAUD = 230400
TPG_PORT = "COM21"
TPG_BAUD = 9600  # Match the TPG366 front panel; 8N1, no flow control.
BASELINE_S = 60.0
TARGETS_C = (30., 40., 50., 60., 70., 80., 90., 100., 110., 120., 130., 140., 150., 160., 170., 175.)
HOLD_S = 300.0  # Continuous thermally qualified observation, not pressure equilibrium.
QUALIFICATION_DEADLINE_S = 600.0
STAGE_DEADLINE_S = 900.0  # Absolute from command; loss of stability cannot extend it.
COOLING_S = 900.0  # T/P observation after verified heater OFF, not proof of being cold.
SAMPLE_S = 1.0  # Best effort; sequential serial reads are not hardware-synchronized.
MAX_SAMPLE_GAP_S = 10.0  # Successful ESI reads occasionally take up to ~5 s; only a longer temperature gap restarts the 60 s window.
OBSERVATION_STALL_S = 60.0  # A longer silence between samples (e.g. the computer slept) stops heating.
HEATER_VOLTAGE_V = 22.0
HEATER_CURRENT_A = 10.0
HEATER_POWER_W = 50.0

import csv
import functools
import hashlib
import importlib.util
import importlib.machinery
import json
import math
import os
import queue
import re
import signal
from pathlib import Path
import sys
import threading
import time
from datetime import datetime, timezone
from uuid import uuid4

NOTEBOOK_DIR = Path(globals().get("__file__", Path.cwd() / "pressure_temperature.ipynb")).resolve().parent
OUTPUT_DIR = NOTEBOOK_DIR / "pressure_temperature_runs"
# Retain uncertain transports across cell re-execution; restarting a kernel is NOT an OFF.
_PT_GUARD = globals().get("_PT_GUARD", {})


def utc(timestamp=None):
    return datetime.fromtimestamp(time.time() if timestamp is None else timestamp, timezone.utc).isoformat()


def json_safe(value):
    if isinstance(value, float) and not math.isfinite(value):
        return None
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    return value


def sync_directory(path):
    if os.name != "nt":
        descriptor = os.open(path, os.O_RDONLY | os.O_DIRECTORY)
        try:
            os.fsync(descriptor)
        finally:
            os.close(descriptor)


def save_json(path, data):
    temporary = path.with_suffix(".tmp")
    with temporary.open("w", encoding="utf-8") as stream:
        json.dump(json_safe(data), stream, indent=2, allow_nan=False)
        stream.flush()
        os.fsync(stream.fileno())
    temporary.replace(path)
    sync_directory(path.parent)


def keep_awake(enabled):
    """Ask Windows to keep the computer and display awake (per thread); a manual sleep is not prevented."""
    try:
        import ctypes
        function = ctypes.WinDLL("kernel32").SetThreadExecutionState
        function.argtypes, function.restype = [ctypes.c_uint], ctypes.c_uint
        ES_CONTINUOUS, ES_SYSTEM_REQUIRED, ES_DISPLAY_REQUIRED = 0x80000000, 0x1, 0x2
        return function(ES_CONTINUOUS | (ES_SYSTEM_REQUIRED | ES_DISPLAY_REQUIRED if enabled else 0)) != 0
    except (AttributeError, OSError):
        return False


def suspended_seconds():
    """Total time this Windows computer has spent asleep or hibernated since boot; None elsewhere."""
    try:
        import ctypes
        kernel32 = ctypes.WinDLL("kernel32")
        kernel32.GetTickCount64.restype = ctypes.c_ulonglong  # Includes sleep/hibernation.
        unbiased = ctypes.c_ulonglong()  # Excludes them, in 100 ns units.
        if not kernel32.QueryUnbiasedInterruptTime(ctypes.byref(unbiased)):
            return None
        return kernel32.GetTickCount64() / 1e3 - unbiased.value / 1e7
    except (AttributeError, OSError):
        return None


def explorer_plugin_dirs():
    """ESIBD Explorer's configured plugin folder (Windows settings), then its default."""
    found = []
    try:
        import winreg
        with winreg.OpenKey(winreg.HKEY_CURRENT_USER, r"Software\ESIBD LAB\ESIBD Explorer\General") as key:
            value = winreg.QueryValueEx(key, "Plugin path")[0]
        if isinstance(value, str) and value and not value.startswith("@"):
            found.append(Path(value).expanduser())
    except (ImportError, OSError):
        pass
    found.append(Path.home() / "ESIBD Explorer" / "plugins")
    return found


def find_plugins(configured=None, notebook_dir=None):
    start = Path(notebook_dir or NOTEBOOK_DIR).resolve()
    candidates = [Path(configured).expanduser().resolve()] if configured is not None else [
        candidate for parent in (start, *start.parents) for candidate in (parent, parent / "plugins")
    ] + explorer_plugin_dirs()
    for root in candidates:
        if ((root / "esi/vendor/runtime/__init__.py").is_file()
                and (root / "tpg366/_runtime/_tpg366.py").is_file()):
            return root
    raise FileNotFoundError("esi/ and tpg366/ not found beside the notebook nor in ESIBD Explorer's plugin folder ("
                            + ", ".join(map(str, explorer_plugin_dirs())) + "). Set PLUGINS_DIR once.")


def load_private(path, package=False, fresh=False):
    path = Path(path).resolve()
    digest = hashlib.sha256(os.path.normcase(str(path)).encode()).hexdigest()[:16]
    pure_helper = path.name in {"_experiment_guard.py", "_heater_stability.py", "_heater_limits.py"}
    prefix = "_esibd_pressure_temperature_helper_" if pure_helper else "_esibd_bundled_pressure_temperature_"
    name = prefix + digest
    if fresh and any(key == name or key.startswith(name + ".") for key in sys.modules):
        raise RuntimeError("ESI runtime already loaded. After confirmed hardware OFF, restart the kernel; never bypass an unfinished-run guard.")
    sources = sorted(path.parent.rglob("*.py")) if package else [path]
    fingerprint = {os.path.normcase(str(p.resolve())): hashlib.sha256(p.read_bytes()).hexdigest() for p in sources}
    if name in sys.modules and getattr(sys.modules[name], "_source_fingerprint", None) != fingerprint:
        raise RuntimeError("Bundled runtime changed after import. Confirm hardware OFF, then restart the kernel.")
    if name not in sys.modules:
        class SourceLoader(importlib.machinery.SourceFileLoader):
            def get_code(self, fullname):
                source = self.get_data(self.path)
                if hashlib.sha256(source).hexdigest() != fingerprint.get(os.path.normcase(str(Path(self.path).resolve()))):
                    raise RuntimeError("Runtime changed during import; restart the kernel.")
                return self.source_to_code(source, self.path)  # Compile verified bytes, never a stale .pyc.

        def source_spec(fullname, file, is_package):
            return importlib.util.spec_from_file_location(
                fullname, file, loader=SourceLoader(fullname, str(file)),
                submodule_search_locations=[str(file.parent)] if is_package else None)

        class PrivateFinder:
            def find_spec(self, fullname, path=None, target=None):
                if not fullname.startswith(name + "."):
                    return None
                relative = fullname[len(name) + 1:].split(".")
                file = package_root.joinpath(*relative)
                is_package = file.is_dir()
                file = file / "__init__.py" if is_package else file.with_suffix(".py")
                return source_spec(fullname, file, is_package) if file.is_file() else None

        package_root = path.parent
        finder = PrivateFinder()
        spec = source_spec(name, path, package)
        if spec is None or spec.loader is None:
            raise ModuleNotFoundError(str(path))
        module = importlib.util.module_from_spec(spec)
        sys.modules[name] = module
        try:
            sys.meta_path.insert(0, finder)  # Restricted to this new private namespace; sys.path is untouched.
            spec.loader.exec_module(module)
            if fingerprint != {os.path.normcase(str(p.resolve())): hashlib.sha256(p.read_bytes()).hexdigest() for p in sources}:
                raise RuntimeError("Runtime changed during import; restart the kernel.")
            module._source_fingerprint = fingerprint
        except BaseException:
            # Keep a failed-load marker: do not reuse partially imported package children.
            module._source_fingerprint = None
            raise
        finally:
            if finder in sys.meta_path:
                sys.meta_path.remove(finder)
    return sys.modules[name]


# Recorded with each run (provenance only; any plugin version may be used).
_RUNTIME_FILES = (
    "esi/vendor/runtime/esi/esi.py",
    "esi/vendor/runtime/esi/esi_base.py",
    "esi/vendor/runtime/_driver_common.py",
    "tpg366/_runtime/_tpg366.py",
    "esi/_heater_stability.py",
    "esi/_experiment_guard.py",
    "esi/_heater_limits.py",
)


def runtime_fingerprints(root):
    """SHA-256 of the runtime files used by this run, for its metadata only."""
    return {path: hashlib.sha256((root / path).read_bytes()).hexdigest()
            for path in _RUNTIME_FILES if (root / path).is_file()}


_SAFE_ESI_REQUIRED = "The loaded ESI runtime lacks the heater safety methods; update the esi/ plugin."


def require_safe_esi(runtime):
    # Check the class actually imported, not merely a candidate path or method names.
    try:
        controller = runtime.ESI._PROCESS_CONTROLLER_CLASS
        valid = (controller.__module__.startswith(runtime.__name__ + ".")
                 and getattr(controller, "HEATER_SAFETY_CONTRACT", 0) >= 1
                 and all(callable(getattr(controller, name, None)) for name in (
                     "_set_heat_module_active_unlocked", "_validate_heat_operating_state_unlocked")))
    except (AttributeError, TypeError) as error:
        raise RuntimeError(_SAFE_ESI_REQUIRED) from error
    if not valid:
        raise RuntimeError(_SAFE_ESI_REQUIRED)


def serial_port_name(value):
    port = str(value).strip()
    match = re.fullmatch(r"(?:COM)?([0-9]+)", port, re.IGNORECASE)
    return f"COM{int(match.group(1))}" if match else port


class CallOwner:
    """One serial call owner. Main-thread Stop never interrupts a native worker's join."""
    PRESSURE_CLOSE_WAIT_S = 4.0  # Waiting budget only; expiry NEVER proves that I/O ended.

    def __init__(self, name):
        self.name = name
        self.jobs = queue.SimpleQueue()
        self.finished = threading.Event()
        self.cleanup = None  # Atomic Python reference; setting it needs no lock or queue operation.
        self.current = None
        self.last_error = None
        self.thread = threading.Thread(target=self._work, name=f"pressure-temperature-{name}", daemon=True)
        self.thread.start()

    def _work(self):
        try:
            while self.cleanup is None:
                try:
                    label, function, result, done = self.jobs.get(timeout=.05)
                except queue.Empty:
                    continue
                self.current = label
                try:
                    if self.cleanup is not None:
                        raise KeyboardInterrupt("Stop requested before queued instrument call")
                    result["value"] = function()
                except BaseException as error:
                    result["error"] = error
                    self.last_error = f"{type(error).__name__}: {error}"
                finally:
                    self.current = None
                    done.set()  # The Python call returned; poison still guards any native timeout.
            self.current = "shutdown"
            self.cleanup()  # Same owner, strictly after the active call, never a second OFF worker.
        except BaseException as error:
            self.last_error = f"{type(error).__name__}: {error}"
        finally:
            self.current = None
            self.finished.set()  # This owner issues no new call; a poisoned native call may still be active.

    def call(self, label, function):
        if self.cleanup is not None:
            raise KeyboardInterrupt("Instrument owner is stopping")
        result, done = {}, threading.Event()
        self.jobs.put((label, function, result, done))
        warned = time.monotonic()
        while not done.wait(.05):
            if self.cleanup is not None:
                raise KeyboardInterrupt("Stop requested; pending I/O remains owned")
            if time.monotonic() - warned >= 5:
                print(f"{self.name}: waiting for {label}; Stop requests safe shutdown, not concurrent Close.")
                warned = time.monotonic()
        if "error" in result:
            raise result["error"]
        return result.get("value")

    def begin_close(self, function):
        if self.cleanup is None:
            self.cleanup = function

    def wait_closed(self, timeout=None, on_interrupt=None):
        started = warned = time.monotonic()
        while not self.finished.is_set():
            try:
                self.finished.wait(.05)
            except KeyboardInterrupt as error:
                if on_interrupt is not None:
                    on_interrupt(error)
                # The queued OFF is already owned. Repeated Stop cannot interrupt it.
            now = time.monotonic()
            if timeout is not None and now - started >= timeout:
                return self.finished.is_set()
            if now - warned >= 5:
                print(f"{self.name} shutdown pending: waiting for the actual call/verified shutdown. Do not rerun or restart to bypass it.")
                warned = now
        return True

    def status(self):
        return {"owner_finished": self.finished.is_set(), "active_call": self.current,
                "last_error": self.last_error, "thread_id": self.thread.ident}


class PressurePort:
    """Own the port even when initialization fails; no reconnect or unit writes."""
    def __init__(self, protocol):
        self.protocol = protocol
        self.port = None
        self.link = None
        self.cancelled = threading.Event()

    def open(self):
        import serial
        if self.cancelled.is_set():
            raise self.protocol.Cancelled("Stopped before pressure port open")
        # Retain the serial object before OS open, including an opening failure or delayed return.
        self.port = serial.Serial(port=None, baudrate=TPG_BAUD, bytesize=8, parity="N", stopbits=1,
                                  timeout=.05, write_timeout=1., xonxoff=False, rtscts=False, dsrdtr=False)
        self.port.port = serial_port_name(TPG_PORT)
        self.port.open()
        if self.cancelled.is_set():
            raise self.protocol.Cancelled("Pressure port opened after Stop; owner will close it without queries")
        self.link = self.protocol.TPG366Link(self.port, self.cancelled, timeout=1.)
        self.link.initialize()
        return {"identification": self.link.identification, "gauges": self.link.gauges}

    def read(self):
        return self.link.read_pressures()  # Already mbar; Reading.unit is the SOURCE unit.

    def close(self):
        self.cancelled.set()
        if self.port is not None:
            self.port.close()
            if self.port.is_open:
                raise RuntimeError("TPG366 port closure unconfirmed")


class LivePlots:
    def __init__(self):
        import matplotlib.pyplot as plt
        self.plt = plt
        self.time_figure, self.axes = plt.subplots(2, 1, sharex=True, figsize=(10, 6), layout="constrained")
        self.temperature_figure, self.temperature_axis = plt.subplots(figsize=(8, 5), layout="constrained")
        self.handles = None
        try:
            from IPython.display import display
            self.handles = [display(fig, display_id=True) for fig in (self.time_figure, self.temperature_figure)]
        except ImportError:
            pass
        # Explicit display updates work with Jupyter's inline backend too.
        for fig in (self.time_figure, self.temperature_figure):
            plt.close(fig)

    def update(self, rows):
        import numpy as np
        for axis in (*self.axes, self.temperature_axis):
            axis.clear()
        t = np.array([row["elapsed_s"] for row in rows]) / 60
        temperature_times = np.array([row["temperature_observed_s"] for row in rows]) / 60
        temperatures = np.array([row["temperature_c"] for row in rows])
        targets = np.array([row["target_temperature_c"] for row in rows])
        for gauge in range(1, 7):
            values = np.array([row[f"P{gauge}_mbar"] for row in rows])
            visible = np.where(values > 0, values, np.nan)  # CSV keeps zero/negative values.
            self.axes[0].plot(t, visible, label=f"G{gauge}", linewidth=1.)
            self.temperature_axis.plot(temperatures, visible, ".", markersize=3, label=f"G{gauge}")
        self.axes[0].set(yscale="log", ylabel="Pressure (mbar)")
        self.axes[0].legend(ncol=6, loc="lower left", bbox_to_anchor=(0, 1.01), borderaxespad=0, fontsize=9)
        self.axes[1].plot(temperature_times, temperatures, label="Measured", color="black")
        self.axes[1].step(temperature_times, targets, where="post", linestyle="--", label="Setpoint", color="tab:red")
        self.axes[1].set(xlabel="Elapsed time (min)", ylabel="Temperature (°C)")
        self.axes[1].legend(loc="upper left")
        self.temperature_axis.set(xlabel="Measured temperature (°C)", ylabel="Pressure (mbar)", yscale="log")
        self.temperature_axis.legend(ncol=6, loc="lower left", bbox_to_anchor=(0, 1.01), borderaxespad=0, fontsize=9)
        for axis in (*self.axes, self.temperature_axis):
            axis.grid(True, alpha=.2)
        for index, fig in enumerate((self.time_figure, self.temperature_figure)):
            fig.canvas.draw()
            if self.handles and self.handles[index] is not None:
                self.handles[index].update(fig)

    def save(self, directory, rows):
        self.update(rows)
        self.time_figure.savefig(directory / "pressure_temperature_time.png", dpi=150)
        self.temperature_figure.savefig(directory / "pressure_vs_temperature.png", dpi=150)


class Experiment:
    """One run; OFF always precedes plotting, metadata finalization and pressure close."""
    def __init__(self, output_dir, esi_factory, pressure_factory, *, guard, operator_restarts=(), baseline_s=BASELINE_S,
                 targets=TARGETS_C, hold_s=HOLD_S, sample_s=SAMPLE_S, overrides=None,
                 qualification_deadline_s=QUALIFICATION_DEADLINE_S, stage_deadline_s=STAGE_DEADLINE_S,
                 cooling_s=COOLING_S, stability_factory=None, limit_preparer=None,
                 clock=time.monotonic, sleep=time.sleep, plots_factory=LivePlots,
                 stall_s=OBSERVATION_STALL_S, awake=keep_awake, suspended=suspended_seconds):
        self.output_dir = Path(output_dir)
        self.guard = guard
        self.hardware_started = self.esi_constructor_attempted = self.pressure_constructor_attempted = False
        self.esi_factory, self.pressure_factory = esi_factory, pressure_factory
        self.baseline_s, self.targets, self.hold_s, self.sample_s = baseline_s, tuple(targets), hold_s, sample_s
        self.overrides = overrides if overrides is not None else {"voltage_v": HEATER_VOLTAGE_V, "current_a": HEATER_CURRENT_A, "power_w": HEATER_POWER_W}
        self.approved_limits = {"voltage_limit_v": HEATER_VOLTAGE_V, "current_limit_a": HEATER_CURRENT_A, "power_limit_w": HEATER_POWER_W}
        self.qualification_deadline_s, self.stage_deadline_s = qualification_deadline_s, stage_deadline_s
        self.cooling_s, self.stability_factory = cooling_s, stability_factory
        self.limit_preparer = limit_preparer
        self.stability = self.stage = self.commanded_stage = None
        self.temperature_observation = None
        self.clock, self.sleep, self.plots_factory = clock, sleep, plots_factory
        self.stall_s, self.awake, self.suspended = stall_s, awake, suspended
        self.last_sample = None
        self.esi = self.pressure = self.plots = self.stream = None
        self.rows, self.errors = [], []
        self.heat_requested = self.connected = self.pressure_healthy = False
        self.shutdown_confirmed = self.pressure_closed = False
        self.pressure_thread = None
        self.pressure_recording_done = threading.Event()
        self.pressure_recording_done.set()
        self.esi_owner = self.pressure_owner = None
        self.stop_pressure = threading.Event()
        self.cancel_heating = threading.Event()
        self.write_lock = threading.Lock()
        self.phase, self.requested, self.applied_target = "preflight", 0., 0.
        self.directory = None
        self.metadata = {"started_utc": utc(), "outcome": "running", "errors": self.errors,
                         "shutdown_confirmed": False, "operator_restarts": list(operator_restarts),
                         "esi_port": ESI_COM, "esi_baud": ESI_BAUD, "tpg_port": serial_port_name(TPG_PORT), "tpg_baud": TPG_BAUD,
                         "baseline_s": baseline_s, "targets_c": self.targets, "hold_s": hold_s,
                         "requested_sample_s": sample_s, "limit_overrides": self.overrides,
                         "approved_heat_ceilings": self.approved_limits,
                         "qualification_deadline_s": qualification_deadline_s, "stage_deadline_s": stage_deadline_s,
                         "cooling_s": cooling_s, "stages": [],
                         "thermal_qualification": "Full 60 s, ±0.2 °C, absolute OLS slope <0.1 °C/min; a temperature gap above stability_parameters.max_gap_s resets the window.",
                         "final_stage": "175 °C hold attempted; accepted band 174.8–175.0 °C; measured >175 °C stops the run.",
                         "residual_state": "Electrical limits remain configured; no NVM save. Cooling observation does not prove cold or safe to touch.",
                         "pressure_unit": "mbar", "temperature_unit": "degC",
                         "timing": "Sequential software observations; no ADC freshness or thermal equilibrium claim.",
                         "observation_continuity": f"Heating stops if the computer sleeps or no sample starts for more than {stall_s:g} s."}

    def validate(self):
        for name, value in (("baseline_s", self.baseline_s), ("hold_s", self.hold_s), ("sample_s", self.sample_s),
                            ("qualification_deadline_s", self.qualification_deadline_s),
                            ("stage_deadline_s", self.stage_deadline_s), ("cooling_s", self.cooling_s),
                            ("stall_s", self.stall_s)):
            if isinstance(value, bool) or not math.isfinite(value) or value <= 0:
                raise ValueError(f"{name} must be finite and positive")
        if not callable(self.limit_preparer):
            raise ValueError("The matched shared electrical-limit helper is required")
        if self.stability_factory is None:
            raise ValueError("The verified shared TemperatureStability helper is required")
        if self.qualification_deadline_s > self.stage_deadline_s:
            raise ValueError("Qualification deadline cannot exceed the absolute stage deadline")
        if (not self.targets or any(isinstance(t, bool) or not math.isfinite(t) or not 0 < t <= 175 for t in self.targets)
                or any(b <= a for a, b in zip(self.targets, self.targets[1:]))):
            raise ValueError("Setpoints must strictly increase and remain within (0, 175] °C")
        for name, value in self.overrides.items():
            if name not in {"voltage_v", "current_a", "power_w"}:
                raise ValueError(f"Unknown heater limit {name}")
            if value is not None and (isinstance(value, bool) or not math.isfinite(value) or value <= 0):
                raise ValueError(f"Explicit {name} must be finite and positive, or None")

    def claim(self):
        if _PT_GUARD.get("run") is not None or (_PT_GUARD.get("guard") not in (None, self.guard)):
            raise RuntimeError("Previous run still owns an uncertain transport; do not rerun. Verify hardware OFF first.")
        self.guard.check()
        self.output_dir.mkdir(parents=True, exist_ok=True)
        sync_directory(self.output_dir.parent)
        self.guard_path = self.output_dir / ".pressure_temperature.running.json"
        _PT_GUARD.update(run=self, guard=self.guard)
        try:
            self.directory = self.output_dir / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ_") + uuid4().hex[:8])
            self.directory.mkdir()
            sync_directory(self.output_dir)
            self.metadata["directory"] = str(self.directory)
            save_json(self.directory / "metadata.json", self.metadata)
            self.stream = (self.directory / "samples.csv").open("x", newline="", encoding="utf-8")
            self.writer = csv.DictWriter(self.stream, fieldnames=list(self.empty_row()))
            self.writer.writeheader()
            self.stream.flush()
            os.fsync(self.stream.fileno())
            sync_directory(self.directory)
            self.metadata["experiment_claim"] = self.guard.claim(self.directory, self.directory / "metadata.json")
            save_json(self.directory / "metadata.json", self.metadata)
        except BaseException as error:
            # Storage preflight failed before constructing/opening any instrument.
            self.fault(error)
            self.metadata.update(outcome="interrupted" if isinstance(error, KeyboardInterrupt) else "error",
                                 ended_utc=utc(), esi_instance_created=False, shutdown_confirmed=False)
            try:
                if self.stream is not None:
                    self.stream.close()
                if self.directory is not None and self.directory.is_dir():
                    save_json(self.directory / "metadata.json", self.metadata)
            finally:
                if self.guard.abandon_before_hardware() is True:
                    _PT_GUARD.clear()
            raise

    def empty_row(self):
        row = {"timestamp_utc": utc(), "elapsed_s": self.clock() - self.started,
               "phase": self.phase, "requested_temperature_c": self.requested,
               "temperature_observed_utc": "", "temperature_observed_s": math.nan,
               "temperature_c": math.nan, "target_temperature_c": math.nan,
               "stage_elapsed_s": math.nan, "thermal_stable": False, "stability_span_s": 0.,
               "slope_c_min": math.nan, "qualified_observation_s": 0., "stage_complete": False,
               "pressure_observed_utc": "", "pressure_source_unit": "", "esi_state_json": "", "error": ""}
        for gauge in range(1, 7):
            row.update({f"P{gauge}_mbar": math.nan, f"G{gauge}_status_code": -1, f"G{gauge}_status": "Unavailable"})
        return row

    def append(self, row):
        with self.write_lock:
            if row["phase"] == "shutdown" and self.stop_pressure.is_set():
                return  # Discard pressure results that reach persistence only after STOP.
            self.writer.writerow(row)
            self.stream.flush()  # Each sample survives a later exception/interrupt.
            self.rows.append(row)

    def fault(self, error):
        self.errors.append(f"{type(error).__name__}: {error}")

    def esi_call(self, method, *args, **kwargs):
        if method in {"collect_diagnostics", "configure_heat_limits", "set_heater_temperature", "set_output_active"}:
            kwargs["cancel_event"] = self.cancel_heating
        def invoke():
            if method in {"set_heater_temperature", "configure_heat_limits"} or (method == "set_output_active" and args[1] is True):
                self.check_stage_deadline()
            if method == "set_heater_temperature" and self.stage is not None:
                self.stage.update(commanded_s=self.clock(), commanded_utc=utc())
                self.commanded_stage = self.stage
            if method == "set_output_active" and args[1] is False and self.commanded_stage is not None:
                self.commanded_stage["heater_off_commanded_s"] = self.clock()
            value = getattr(self.esi, method)(*args, **kwargs)
            if method == "collect_diagnostics":
                self.temperature_observation = (self.clock(), utc())  # Actual owner return, before persistence/rendering.
            if method == "connect" and value is True:
                self.connected = True  # Retain late success even if the waiting cell was interrupted.
            return value
        return self.esi_owner.call(method, invoke)

    def cancel_pressure(self):
        if self.pressure is not None:
            self.pressure.cancelled.set()  # Event only: no serial I/O on the caller thread.

    def request_stop(self):
        self.cancel_heating.set()  # Cancel readiness/positive commands, never the owned OFF cleanup.
        # Queue OFF before any subsequent CSV/plot/export work. It follows, never overlaps, the active call.
        if self.esi_owner is not None:
            self.esi_owner.begin_close(self.stop_esi)
        if not self.pressure_healthy:
            self.cancel_pressure()

    def note_interrupt(self, error):
        self.fault(error)
        if self.metadata["outcome"] == "complete":
            self.metadata["outcome"] = "interrupted"

    def transport_poison_state(self):
        try:
            state = self.esi._transport_poisoned
            return state if type(state) is bool else None
        except BaseException:
            return None

    def transport_is_safe_to_call(self):
        state = self.transport_poison_state()
        if state is not False:
            reason = "poisoned" if state is True else "unknown"
            self.fault(RuntimeError(f"ESI transport {reason}: no further DLL call, OFF or Close attempted"))
            return False
        return True

    def snapshot_error(self, diagnostic):
        heat = diagnostic["heat"]
        if diagnostic["main_state"]["name"] != "STATE_ON" or int(diagnostic["device_state"]["hex"], 16) != 0:
            return "ESI controller not ready; no interlock/configuration bypass is attempted"
        for address in (1, 2):
            hv = diagnostic["modules"][address]
            if hv["module_active"] or hv["target_v"] != 0:
                return f"HV module {address} is not confirmed OFF with zero target"
        temperature = heat["monitor_temperature_c"]
        maximum = heat["hardware_limits"]["max_temperature_c"]
        if not (heat["valid"] is True and not isinstance(temperature, bool) and not isinstance(maximum, bool)
                and math.isfinite(temperature) and math.isfinite(maximum)
                and 0 <= temperature <= min(175., maximum)):
            return "Heater sensor unavailable or temperature outside the verified 0–175 °C window"
        target = heat["target_temperature_c"]
        if not math.isfinite(target) or not math.isclose(target, self.applied_target, rel_tol=1e-9, abs_tol=1e-6):
            return "Heater setpoint changed or unverified"
        configured = self.metadata.get("applied_heat_configuration", {})
        for key, ceiling in self.approved_limits.items():
            if key in configured:
                maximum = heat["hardware_limits"]["max_" + key.replace("_limit", "")]
                if (isinstance(heat[key], bool) or isinstance(maximum, bool)
                        or heat[key] != configured[key] or not math.isfinite(maximum)
                        or not 0 < heat[key] <= min(ceiling, maximum)):
                    return "Heater electrical limits changed or exceed approved/hardware ceilings"
        if bool(heat["module_active"]) != self.heat_requested:
            return "Heater activation state disagrees with the run"
        if self.heat_requested and not heat["active"]:
            return "Heater regulation is not active"
        return None

    def check_continuity(self):
        # Nothing is observed while the computer sleeps or the notebook stalls; the instruments ran unsupervised.
        now, asleep_total, stamp = self.clock(), self.suspended(), utc()
        previous, self.last_sample = self.last_sample, (now, asleep_total, stamp)
        if previous is None:
            return
        gap = now - previous[0]
        asleep = None if asleep_total is None or previous[1] is None else asleep_total - previous[1]
        if gap <= self.stall_s and (asleep is None or asleep <= 1.):
            return
        self.metadata.setdefault("observation_interruptions", []).append(
            {"from_utc": previous[2], "to_utc": stamp, "gap_s": gap, "suspended_s": asleep})
        if asleep is not None and asleep > 1.:
            raise RuntimeError(f"The computer slept or hibernated for {asleep:.0f} s ({previous[2]} to {stamp}); "
                               "nothing was observed meanwhile, so heating stops. Keep the PC awake during runs.")
        raise RuntimeError(f"No sample for {gap:.0f} s ({previous[2]} to {stamp}): the computer probably slept "
                           "or the notebook stalled; nothing was observed meanwhile, so heating stops.")

    def sample(self, *, pressure_only=False):
        row = self.empty_row()
        problem = None
        try:
            if not pressure_only:
                self.check_continuity()
            self.pressure_healthy = False
            reading = self.pressure_owner.call("read_pressures", self.pressure.read)
            row["pressure_observed_utc"], row["pressure_source_unit"] = utc(reading.received_at), reading.unit
            if len(reading.pressures) != 6 or len(reading.statuses) != 6:
                raise RuntimeError("Pressure frame does not contain six gauges")
            for gauge, (value, status) in enumerate(zip(reading.pressures, reading.statuses), 1):
                if status not in range(7) or (status == 0 and not math.isfinite(value)):
                    raise RuntimeError("Invalid gauge status/measurement")
                row[f"P{gauge}_mbar"] = value if status == 0 else math.nan
                row[f"G{gauge}_status_code"] = status
                row[f"G{gauge}_status"] = ("OK", "Underrange", "Overrange", "Sensor error", "Sensor off", "No sensor", "Identification error")[status]
            # Acquisition only: no implicitly critical gauge or vacuum interlock.
            if not any(status == 0 for status in reading.statuses):
                raise RuntimeError("No valid pressure among the six gauges; heating stopped")
            self.pressure_healthy = True
            if not pressure_only:
                diagnostic = self.esi_call("collect_diagnostics")
                observed, row["temperature_observed_utc"] = self.temperature_observation
                row["temperature_observed_s"] = observed - self.started
                row["esi_state_json"] = json.dumps(json_safe(diagnostic), allow_nan=False)
                heat = diagnostic["heat"]
                if (heat["valid"] is True and not isinstance(heat["monitor_temperature_c"], bool)
                        and math.isfinite(heat["monitor_temperature_c"])):
                    row["temperature_c"] = heat["monitor_temperature_c"]
                row["target_temperature_c"] = heat["target_temperature_c"]
                message = self.snapshot_error(diagnostic)
                if message:
                    raise RuntimeError(message)
                if self.phase == "heating" and self.stage is not None:
                    self.qualify(row, observed)
        except BaseException as error:
            problem = error
            if not pressure_only:
                self.request_stop()
            row["error"] = f"{type(error).__name__}: {error}"
        if pressure_only and self.stop_pressure.is_set():
            return None  # A late reply cannot change the finalized CSV/metadata after a blocked shutdown.
        self.append(row)
        if problem is not None:
            raise problem
        return row

    def qualify(self, row, observed):
        stage = self.stage
        elapsed = observed - stage["commanded_s"]
        row["stage_elapsed_s"] = elapsed
        state = self.stability.update(observed, row["temperature_c"], self.requested,
                                      active=self.heat_requested, valid=True)
        row.update(thermal_stable=state["stable"], stability_span_s=state["span_s"],
                   slope_c_min=state["slope_c_min"] if state["slope_c_min"] is not None else math.nan)
        stage["last_observed_s"] = observed
        stage["last_stability"] = state
        if elapsed < 0 or elapsed > self.stage_deadline_s:
            raise RuntimeError("Absolute stage deadline exceeded; stage incomplete")
        if stage["first_qualified_s"] is None and elapsed > self.qualification_deadline_s:
            raise RuntimeError("First thermal qualification deadline exceeded; stage incomplete")
        if state["stable"]:
            if stage["first_qualified_s"] is None:
                stage["first_qualified_s"] = observed
            if stage["observation_started_s"] is None:
                stage["observation_started_s"] = observed
                print(f"{self.requested:g} °C qualified; observing continuously for {self.hold_s:g} s.")
            stage["qualified_observation_s"] = observed - stage["observation_started_s"]
        else:
            if stage["observation_started_s"] is not None:
                stage["observation_resets"] = stage.get("observation_resets", 0) + 1
                print(f"{self.requested:g} °C qualification lost; the absolute stage deadline is unchanged.")
            stage["observation_started_s"] = None
            stage["qualified_observation_s"] = 0.
        row["qualified_observation_s"] = stage["qualified_observation_s"]
        stage["complete"] = bool(state["stable"] and stage["qualified_observation_s"] >= self.hold_s)
        row["stage_complete"] = stage["complete"]
        if stage["complete"]:
            stage["completed_s"] = observed
        elif elapsed >= self.stage_deadline_s:
            raise RuntimeError("Absolute stage deadline reached; stage incomplete")
        elif stage["first_qualified_s"] is None and elapsed >= self.qualification_deadline_s:
            raise RuntimeError("First thermal qualification deadline reached; stage incomplete")

    def check_stage_deadline(self, stage=None, *, at=None):
        stage = self.commanded_stage if stage is None else stage
        now = self.clock() if at is None else at
        if stage is not None and now > stage["commanded_s"] + self.stage_deadline_s:
            stage.update(complete=False, deadline_exceeded_s=now)
            raise RuntimeError("Absolute stage deadline exceeded; no further heating command is allowed")

    def observe_stage(self):
        next_sample, last_plot = self.clock(), -math.inf
        while True:
            budget = self.stage_deadline_s if self.stage["first_qualified_s"] is not None else self.qualification_deadline_s
            deadline = self.stage["commanded_s"] + budget
            if self.clock() > deadline:
                raise RuntimeError("Thermal qualification/stage deadline exceeded; stage incomplete")
            self.sleep(max(0., min(next_sample, deadline) - self.clock()))
            self.sample()
            # Genuine observations do not authorize continued heating after stalled persistence.
            self.check_stage_deadline(self.stage)
            if self.stage["complete"]:
                return
            if self.clock() - last_plot >= 5:
                self.plots.update(self.rows)
                last_plot = self.clock()
            next_sample = max(next_sample + self.sample_s, self.clock())

    def cool_down(self):
        # This is observation, not a safety confirmation; Stop/fault always jumps to shutdown.
        if self.esi_call("set_output_active", 0, False) is not False:
            raise RuntimeError("Heater OFF unconfirmed; cooling observation refused")
        self.heat_requested, self.requested, self.applied_target = False, 0., 0.
        if self.commanded_stage is not None:
            # Always send OFF first, even when work between completion and OFF exceeded the budget.
            self.check_stage_deadline(at=self.commanded_stage["heater_off_commanded_s"])
        self.phase = "cooling"
        self.metadata["cooling_started_s"] = self.clock()
        self.metadata["cooling_started_utc"] = utc()
        self.metadata["cooling_complete"] = False
        print(f"Heater OFF verified; observing temperature and pressure for {self.cooling_s:g} s. This does not prove cold.")
        self.sample()  # Fresh sensor + module/HV/gates/zero setpoint checks after verified OFF.
        self.dwell(self.cooling_s, started=self.metadata["cooling_started_s"])
        self.metadata["cooling_complete"] = True
        self.metadata["cooling_finished_s"] = self.clock()

    def dwell(self, duration, *, started=None):
        deadline = (self.clock() if started is None else started) + duration
        next_sample = self.clock() + self.sample_s
        last_plot = -math.inf
        while self.clock() < deadline:
            self.sleep(max(0., min(next_sample, deadline) - self.clock()))
            if self.clock() >= deadline:
                break
            self.sample()
            if self.clock() - last_plot >= 5:
                self.plots.update(self.rows)
                last_plot = self.clock()
            # No catch-up burst or fictitious equally spaced timestamps after slow reads.
            next_sample = max(next_sample + self.sample_s, self.clock())

    def preflight(self):
        if self.esi_call("connect") is not True:
            raise RuntimeError("ESI connection unconfirmed")
        diagnostic = self.esi_call("collect_diagnostics")
        self.metadata["initial_diagnostics"] = diagnostic
        message = self.snapshot_error(diagnostic)
        if message:
            raise RuntimeError(message)
        config = self.esi_call("get_heat_configuration")
        self.metadata["initial_heat_configuration"] = config
        print(f"Existing heater limits: {config['voltage_limit_v']:g} V / {config['current_limit_a']:g} A / {config['power_limit_w']:g} W.")
        maximum = config["hardware_limits"]["max_temperature_c"]
        if not math.isfinite(maximum) or maximum < 175 or max(self.targets) > maximum:
            raise ValueError(f"Requested program exceeds getter Tmax = {maximum!r} °C")
        wanted = {}
        for short, long, unit in (("voltage_v", "voltage", "v"), ("current_a", "current", "a"), ("power_w", "power", "w")):
            value = self.overrides.get(short)
            value = config[f"{long}_limit_{unit}"] if value is None else value
            top = config["hardware_limits"][f"max_{long}_{unit}"]
            if (self.overrides.get(short) is not None and not isinstance(top, bool)
                    and isinstance(top, (int, float)) and math.isfinite(top) and top > 0):
                value = min(value, top)
            if not (not isinstance(value, bool) and not isinstance(top, bool)
                    and math.isfinite(value) and math.isfinite(top)
                    and 0 < value <= min(top, self.approved_limits[f"{long}_limit_{unit}"])):
                raise ValueError(f"Heater {short} is {value!r}: set a positive load-appropriate limit or its explicit override before heating. Hardware maxima are NOT defaults.")
            wanted[f"{long}_limit_{unit}"] = value
        self.metadata["requested_heat_limits"] = wanted
        expected = dict(wanted)
        if any(value is not None for value in self.overrides.values()):
            bounded = {}
            for short, long, unit in (("voltage_v", "voltage", "v"), ("current_a", "current", "a"), ("power_w", "power", "w")):
                if self.overrides.get(short) is None:
                    continue
                ceiling = wanted[f"{long}_limit_{unit}"]
                value = self.limit_preparer(ceiling, power=(unit == "w"))
                if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value) or not 0 < value <= ceiling:
                    raise RuntimeError("Prepared native limit input is invalid or exceeds its ceiling")
                bounded[short] = value
            self.metadata["prepared_heat_limit_inputs"] = bounded
            echoes = self.esi_call("configure_heat_limits", **bounded)
            self.metadata["applied_heat_limit_echoes"] = echoes
            if not isinstance(echoes, dict) or set(echoes) != {key for key, value in self.overrides.items() if value is not None}:
                raise RuntimeError("Heater electrical limit readback mismatch: missing applied echoes")
            for short, long, unit in (("voltage_v", "voltage", "v"), ("current_a", "current", "a"), ("power_w", "power", "w")):
                if self.overrides.get(short) is not None:
                    value = echoes.get(short)
                    top = config["hardware_limits"][f"max_{long}_{unit}"]
                    if (isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value)
                            or not 0 < value <= min(top, self.approved_limits[f"{long}_limit_{unit}"])):
                        raise RuntimeError("Applied heater electrical limit is missing, invalid or exceeds approved/hardware ceilings")
                    expected[f"{long}_limit_{unit}"] = value
        applied = self.esi_call("get_heat_configuration")
        self.metadata["applied_heat_configuration"] = applied
        for key, value in expected.items():
            measured = applied[key]
            maximum = applied["hardware_limits"]["max_" + key.replace("_limit", "")]
            if not (not isinstance(measured, bool) and not isinstance(maximum, bool)
                    and math.isfinite(measured) and math.isfinite(maximum)
                    and 0 < measured <= min(maximum, self.approved_limits[key])
                    and math.isclose(measured, value, rel_tol=1e-9, abs_tol=1e-6)):
                raise RuntimeError("Heater electrical limit readback mismatch; heating refused")
        self.metadata["pressure_identity"] = self.pressure_owner.call("open", self.pressure.open)
        self.phase = "baseline"
        first = self.sample()
        self.metadata["initial_unheated_temperature_c"] = first["temperature_c"]
        if first["temperature_c"] > self.targets[0]:
            raise ValueError(f"Initial temperature {first['temperature_c']:.10g} °C exceeds the first target ({self.targets[0]:g} °C); "
                             f"heating refused: let the instrument cool below {self.targets[0]:g} °C, then rerun. No stages are skipped.")
        self.program = list(self.targets)
        self.metadata["program_c"] = self.program
        save_json(self.directory / "metadata.json", self.metadata)
        print(f"Preflight: HV OFF; initial unheated temperature {first['temperature_c']:.2f} °C; applied limits {expected}; targets {self.program} °C.")

    def stop_esi(self):
        # Executed ONLY by the original ESI owner, after its pending call really returned.
        if self.esi is None:
            self.shutdown_confirmed = not self.esi_constructor_attempted
            return
        if not self.transport_is_safe_to_call():
            return
        if self.connected:
            try:
                if self.esi.set_output_active(0, False) is not False:
                    raise RuntimeError("Heater OFF readback unconfirmed")
            except BaseException as error:
                self.fault(error)
        if self.transport_is_safe_to_call():
            try:
                closed = self.esi.disconnect(on_discharge=self.discharge_observation)
                self.shutdown_confirmed = closed is True and self.connected
                if not self.shutdown_confirmed:
                    raise RuntimeError("ESI OFF/discharge/port closure not confirmed for an established connection")
            except BaseException as error:
                self.fault(error)

    def discharge_observation(self, state):
        # Keep every discharge round, not only the last, as evidence for the unexplained HV ADC readings.
        observations = self.metadata.setdefault("discharge_observations", [])
        if len(observations) < 2000:
            observations.append({"elapsed_s": self.clock() - self.started, **state})
        self.metadata["last_discharge"] = state

    def close_pressure(self):
        # Same owner as Serial(), initialize() and read(); also runs after a late OS open.
        try:
            if self.pressure is not None:
                self.pressure.close()
            elif self.pressure_constructor_attempted:
                raise RuntimeError("TPG constructor failed; pressure closure is unconfirmed")
            self.pressure_closed = True
        except BaseException as error:
            self.fault(error)

    def shutdown(self):
        self.heat_requested, self.requested, self.phase = False, 0., "shutdown"
        self.request_stop()
        def record_pressure():
            try:
                while not self.stop_pressure.is_set():
                    self.sample(pressure_only=True)
                    self.stop_pressure.wait(self.sample_s)
            except BaseException as error:
                self.fault(error)  # Independent pressure failure cannot postpone the queued ESI OFF.
            finally:
                self.pressure_recording_done.set()
        if self.pressure_healthy:
            try:
                self.pressure_recording_done.clear()
                self.pressure_thread = threading.Thread(target=record_pressure, daemon=True)
                self.pressure_thread.start()
            except BaseException as error:
                self.fault(error)
        if self.esi_owner is not None:
            # No short notebook timeout may kill the runtime's verified discharge (up to 60 s).
            self.esi_owner.wait_closed(on_interrupt=self.note_interrupt)
        else:
            self.shutdown_confirmed = self.esi is None
        self.stop_pressure.set()
        self.cancel_pressure()
        if self.pressure_owner is not None:
            self.pressure_owner.begin_close(self.close_pressure)
            pressure_idle = self.pressure_owner.wait_closed(
                timeout=CallOwner.PRESSURE_CLOSE_WAIT_S, on_interrupt=self.note_interrupt)
        else:
            pressure_idle = self.pressure_closed = self.pressure is None
        if self.pressure_thread is not None:
            deadline = time.monotonic() + CallOwner.PRESSURE_CLOSE_WAIT_S
            while not self.pressure_recording_done.is_set() and time.monotonic() < deadline:
                try:
                    self.pressure_recording_done.wait(.05)
                except KeyboardInterrupt as error:
                    self.note_interrupt(error)
            # Never trust Thread.is_alive() after an interrupted join; use the target's own completion event.
            pressure_idle = pressure_idle and self.pressure_recording_done.is_set()
        self.metadata["pressure_recording_finished"] = self.pressure_recording_done.is_set()
        if not pressure_idle:
            self.fault(RuntimeError("Pressure owner or recorder still active; no concurrent serial Close; rerun blocked"))
        self.metadata["esi_transport_poisoned"] = self.transport_poison_state() if self.esi is not None else None
        self.metadata["io_owners"] = {name: owner.status() for name, owner in (
            ("esi", self.esi_owner), ("pressure", self.pressure_owner)) if owner is not None}
        return pressure_idle

    def run(self):
        previous = None
        if threading.current_thread() is threading.main_thread():
            previous = signal.getsignal(signal.SIGINT)
            def stop_from_signal(signum, frame):
                # Cooperative cancellation only: no instrument I/O or wait for the native call here.
                self.cancel_heating.set()
                if self.esi_owner is not None:
                    self.esi_owner.begin_close(self.stop_esi)
                raise KeyboardInterrupt("User Stop; queued verified shutdown after the active call")
            signal.signal(signal.SIGINT, stop_from_signal)
        try:
            self.metadata["keep_awake"] = self.awake(True) is True
            if not self.metadata["keep_awake"] and sys.platform == "win32":
                print("Windows refused the keep-awake request: disable sleep/hibernation in the power plan.")
            return self.run_sequence()
        finally:
            if previous is not None:
                signal.signal(signal.SIGINT, previous)
            self.awake(False)

    def run_sequence(self):
        self.validate()
        self.started = self.clock()
        self.claim()  # The CSV, metadata and persistent guard exist before hardware access.
        try:
            self.plots = self.plots_factory()
            self.esi_owner = CallOwner("ESI")
            self.pressure_owner = CallOwner("TPG366")
            def create_pressure():
                self.pressure_constructor_attempted = True
                self.pressure = self.pressure_factory()
            def create_esi():
                self.esi_constructor_attempted = True
                self.esi = self.esi_factory(self.directory)
            self.hardware_started = True  # Intent recorded before the irreversible guard transition.
            self.guard.mark_hardware_started()
            self.pressure_owner.call("construct", create_pressure)
            self.esi_owner.call("construct", create_esi)
            self.preflight()
            self.dwell(self.baseline_s)
            self.phase = "heating"
            for target in self.program:
                self.guard.check()
                # Recheck gates and limits before a new demand; never re-enable after a fault.
                message = self.snapshot_error(self.esi_call("collect_diagnostics"))
                if message:
                    raise RuntimeError(message)
                self.requested = target
                print(f"Stage {target:g} °C: first qualification within {self.qualification_deadline_s:g} s; total deadline {self.stage_deadline_s:g} s.")
                self.stability = self.stability_factory()
                self.metadata["stability_parameters"] = {name: getattr(self.stability, name) for name in
                    ("window_s", "tolerance_c", "drift_c_min", "max_gap_s")}
                self.stage = {"target_c": target, "complete": False, "first_qualified_s": None,
                              "observation_started_s": None, "qualified_observation_s": 0.}
                self.metadata["stages"].append(self.stage)
                self.applied_target = self.esi_call("set_heater_temperature", target)
                self.stage["applied_target_c"] = self.applied_target
                if (isinstance(self.applied_target, bool) or not math.isfinite(self.applied_target)
                        or not 0 < self.applied_target <= min(target, 175.)
                        or self.applied_target != target):
                    # All integer stage targets are exactly represented by the matched DLL.
                    raise RuntimeError("Applied heater setpoint differs from the requested integer stage; heating refused")
                if not self.heat_requested:
                    self.heat_requested = True  # Mark the intent before the possibly failing ON call.
                    if self.esi_call("set_output_active", 0, True) is not True:
                        raise RuntimeError("Heater ON unconfirmed")
                self.observe_stage()
            self.guard.check()
            self.cool_down()
            self.metadata["outcome"] = "complete"
        except BaseException as error:
            self.request_stop()
            self.fault(error)
            self.metadata["outcome"] = "interrupted" if isinstance(error, KeyboardInterrupt) else "error"
            # Say why now: the verified shutdown that follows can take about a minute.
            print(f"Run stopped: {type(error).__name__}: {error}\n"
                  "Shutting down: heater OFF, then the ESI discharge check and port closure (up to about 60 s).")
        finally:
            pressure_idle = self.shutdown()
            if not self.shutdown_confirmed or not self.pressure_closed or not pressure_idle:
                self.metadata["outcome"] = "shutdown_unconfirmed"
                if not self.esi_constructor_attempted:
                    print("ESI was not accessed; pressure ownership/closure unconfirmed. Rerun blocked.")
                elif not self.shutdown_confirmed:
                    print("SHUTDOWN UNCONFIRMED: heater/HV may remain energized. Use the hardware safety controls; rerun blocked.")
                elif not self.pressure_closed:
                    print("ESI OFF/discharge verified; pressure link closure unconfirmed. Rerun blocked.")
                else:
                    print("ESI OFF/discharge and pressure link closure verified; pressure recording still pending. Rerun blocked.")
            if pressure_idle and self.stream is not None:
                try:
                    self.stream.close()
                except BaseException as error:
                    self.fault(error)
            if self.errors and self.metadata["outcome"] == "complete":
                self.metadata["outcome"] = "error"
            self.metadata.update(ended_utc=utc(), samples=len(self.rows), esi_instance_created=self.esi is not None,
                                 shutdown_confirmed=self.shutdown_confirmed if self.esi is not None else False,
                                 pressure_closed=self.pressure_closed, hardware_started=self.hardware_started,
                                 pressure_nak_retransmissions=getattr(getattr(self.pressure, "link", None), "nak_count", None))
            save_json(self.directory / "metadata.json", self.metadata)
            try:
                if self.hardware_started:
                    owners_idle = bool(pressure_idle and self.esi_owner is not None and self.esi_owner.finished.is_set())
                    cleared = self.guard.finalize(shutdown_confirmed=self.shutdown_confirmed,
                        owners_idle=owners_idle, auxiliary_closed=self.pressure_closed,
                        transport_poisoned=self.transport_poison_state() if self.esi is not None else None)
                else:
                    cleared = self.guard.abandon_before_hardware()
            except BaseException as error:
                self.fault(error)
                cleared = False
                self.metadata["outcome"] = "shutdown_unconfirmed"
                self.metadata["guard_release_error"] = f"{type(error).__name__}: {error}"
                save_json(self.directory / "metadata.json", self.metadata)
            if cleared is True:
                _PT_GUARD.clear()
            else:
                print("Experiment ownership retained; do not rerun. ESI shutdown and pressure closure remain reported separately.")
            try:
                if self.plots is not None:
                    self.plots.save(self.directory, list(self.rows))
            except BaseException as error:
                self.fault(error)
                if self.metadata["outcome"] == "complete":
                    self.metadata["outcome"] = "error"
                save_json(self.directory / "metadata.json", self.metadata)
            if self.errors:
                print("\n".join(self.errors))
            print(f"{self.metadata['outcome']}: {len(self.rows)} samples → {self.directory}")
        return self


def main():
    if not ARM_HEATING:
        print("Heating not armed. No hardware accessed. Review ports/limits, then set ARM_HEATING=True.")
        if _PT_GUARD or (OUTPUT_DIR / ".pressure_temperature.running.json").exists():
            print("Previous shutdown still unconfirmed: disarming this cell does not switch hardware OFF.")
        return None
    if _PT_GUARD:
        raise RuntimeError("Unfinished run in this kernel. Use hardware safety controls and terminate old owning processes before operator recovery in a fresh kernel.")
    if sys.platform != "win32":
        raise RuntimeError("ESI requires Windows and the bundled vendor DLL. No hardware accessed.")
    if isinstance(ESI_COM, bool) or not isinstance(ESI_COM, int) or not 1 <= ESI_COM <= 255:
        raise ValueError("ESI_COM must be a COM number from 1 to 255")
    if not TPG_PORT or ESI_BAUD != 230400 or TPG_BAUD not in (9600, 19200, 38400, 57600, 115200):
        raise ValueError("Check ports and configured baud rates; no ESI baud fallback is attempted")
    if serial_port_name(TPG_PORT).upper().removeprefix("\\\\.\\") == f"COM{ESI_COM}":
        raise ValueError("ESI and TPG366 must not share a COM port")
    root = find_plugins(PLUGINS_DIR)
    fingerprints = runtime_fingerprints(root)
    guards = load_private(root / "esi/_experiment_guard.py")
    guard = guards.ExperimentGuard(plugin_dir=root / "esi", com=ESI_COM, kind="pressure_temperature",
                                   output_dir=OUTPUT_DIR, kernel_globals=globals(),
                                   legacy_pt_dirs=(NOTEBOOK_DIR / "pressure_temperature_runs",))
    run = None
    try:
        records = guard.authorize_restart()
        _PT_GUARD["guard"] = guard
        runtime = load_private(root / "esi/vendor/runtime/__init__.py", package=True, fresh=True)
        require_safe_esi(runtime)
        protocol = load_private(root / "tpg366/_runtime/_tpg366.py")
        stability = load_private(root / "esi/_heater_stability.py")
        limits = load_private(root / "esi/_heater_limits.py")
        run = Experiment(OUTPUT_DIR,
                         lambda directory: runtime.ESI("pressure-temperature", ESI_COM, baudrate=ESI_BAUD,
                                                        log_dir=directory, process_backend=False),
                         lambda: PressurePort(protocol), guard=guard, operator_restarts=records,
                         stability_factory=functools.partial(stability.TemperatureStability, max_gap_s=MAX_SAMPLE_GAP_S),
                         limit_preparer=limits.bounded_limit)
        run.metadata["runtime_sources"] = fingerprints
        return run.run()
    except BaseException:
        if run is None or (not run.hardware_started and _PT_GUARD.get("guard") is guard):
            if guard.abandon_before_hardware() is True:
                if _PT_GUARD.get("guard") is guard:
                    _PT_GUARD.clear()
            else:
                _PT_GUARD["guard"] = guard
        raise


main()
